# Using the Model

In this notebook, we will evaluate a given tuple of architecture, workload, and mapping using the AccelForge *model*. In the `mapper.ipynb` notebook, we will see how the AccelForge *mapper* can automatically explore the space of mappings (*i.e.*, the *mapspace*) to find the optimal mapping for a given architecture and workload.

## Unfused Mapping Example

The cell below loads a simple architecture, a chain of matrix multiplications,
and an unfused mapping.

In [ ]:
import accelforge as af

spec = af.Spec.from_yaml(
    af.examples.arches.simple,
    af.examples.workloads.basic.matmuls,
    af.examples.mappings.unfused_matmuls_to_simple,
    jinja_parse_data={
        "N_EINSUMS": 2,
        "M": 64,
        "KN": 32,
        "MainMemoryEnergy": 40,
        "GlobalBufferSize": 1e5,
    },
)

Accessing the architecture spec in a cell displays a diagram of the architecture:

In [ ]:
spec.arch

Similarly, the cell below displays a diagram of the workload Einsums:

In [ ]:
spec.workload

Finally, we display the mapping in the LoopTree notation below.

In [ ]:
spec.mapping

Using the `spec.evaluate_mapping()` method, we can call the AccelForge model, which returns a result object from which we can access metrics such as energy and latency.

In [ ]:
result = spec.evaluate_mapping()
print("Energy:", result.energy())
print("Latency:", result.latency())
print()
print("Energy breakdown:")
print(result.energy(per_component=True))

The results object also contains the memory usage of each level of the hierarchy. The memory usage is reported as a number between 0 and 1 representing the fraction of the memory capacity used. Note that the usage of MainMemory is zero because we have set the capacity to be infinite.

In [ ]:
print(result.resource_usage())

AccelForge comes with visualization tools. For example, we can show the memory usage during the processing of Matmul0 and Matmul1 broken down by tensors using the code below:

In [ ]:
from accelforge.plotting.mappings import plot_memory_usage_breakdown

plot_memory_usage_breakdown([result], memory_levels=["GlobalBuffer"])

The plot above makes it easy to gain insights into how the mapping is utilizing hardware resources. For example, we can see that tensors $T0$ and $T1$ represent the largest memory usage.

Below, we plot the energy consumption broken down by components in the architecture.

In [ ]:
from accelforge.plotting.mappings import plot_energy_breakdown

plot_energy_breakdown([result], separate_by=["component"])

We can see that GlobalBuffer accesses make up the greatest portion of energy consumption, but the energy consumption of MainMemory is also significant.

The visualization tool makes it easy to further breakdown energy consumption by Einsum:

In [ ]:
plot_energy_breakdown([result], separate_by=["component"], stack_by=["einsum"])

The plot above shows that Matmul0 and Matmul1 consumes roughly the same amount of energy.

Finally, we break down the energy consumption by tensor. Note that energy consumption of compute units does not break down by tensors, thus the MAC energy remains as a single bar.

In [ ]:
plot_energy_breakdown([result], separate_by=["component"], stack_by=["tensor"])

We can see that accesses of tensor $T1$ consumes a lot of energy.

## Fused Mapping Example

We'll now parse in a simple fused mapping.

In [ ]:
spec = af.Spec.from_yaml(
    af.examples.arches.simple,
    af.examples.workloads.basic.matmuls,
    af.examples.mappings.fused_matmuls_to_simple,
    jinja_parse_data={
        "N_EINSUMS": 2,
        "M": 64,
        "KN": 32,
        "MainMemoryEnergy": 30,
        "GlobalBufferSize": 1e5,
    },
)

The cell below shows the mapping, now fused.

In [ ]:
spec.mapping

We call the model as before:

In [ ]:
fused_result = spec.evaluate_mapping()
print("Energy:", fused_result.energy())
print("Latency:", fused_result.latency())
print()
print("Energy breakdown:")
print(result.energy(per_component=True))

And we can visualize the results using the same functions.

In [ ]:
plot_memory_usage_breakdown([fused_result], memory_levels=["GlobalBuffer"])

In [ ]:
from accelforge.plotting.mappings import plot_energy_breakdown

plot_energy_breakdown([fused_result], separate_by=["component"])

Interestingly, MainMemory is no longer the 

In [ ]:
plot_energy_breakdown([fused_result], separate_by=["component"], stack_by=["einsum"])

In [ ]:
plot_energy_breakdown([fused_result], separate_by=["component"], stack_by=["tensor"])

Finally, we'll do an energy comparison between the unfused and fused mappings.

In [ ]:
plot_energy_breakdown(
    [result, fused_result],
    separate_by=["component"],
    stack_by=["tensor"],
    labels=["Unfused", "Fused"],
)